# Serving the final model on the CPU
Hosting NLLB-200 1.3B on a GPU is the expensive part of serving it. This notebook merges the adapter
of the final model (§7.4 of `sicilian_nmt.ipynb`) into the base model, converts it to CTranslate2
with int8 weights (`experiments/serving/ct2_export.py`, saved to `nllb-final-ct2-int8` on Drive)
and measures it on the CPU of the runtime with 1, 2 and 4 threads: time to load, resident memory
and latency of one sentence at a time (beam 5) on 100 English sentences of our test set translated
into Sicilian. It then translates the 8 test files of §7.9 with int8 weights and scores them next
to the fp16 GPU hypotheses of that cell, raw and in the tokenized space of Wdowiak, so that the
cost of the quantization is measured on the same lines. Results go to `results_ct2.json`, and a
finished step is skipped. The merge needs about 12 GB of RAM, so a GPU runtime (for its memory
and its CPUs, not for the GPU) is the safe choice.

In [3]:
!pip -q install transformers==4.57.6 peft==0.17.1 ctranslate2 sentencepiece sacrebleu
import os, sys, json
if not os.path.exists('SicilianNMT'):
    !git clone -q https://github.com/dmeoli/SicilianNMT.git
!git -C SicilianNMT pull -q
from google.colab import drive; drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/SicilianNMT-colab'
sys.path.append('SicilianNMT/experiments/serving')
import ct2_export
print('ready')

Mounted at /content/drive
ready


In [4]:
# --- CTranslate2 int8 export of the final model and CPU benchmark ---
from glob import glob
ep = json.load(open(f'{OUT}/results_final.json'))['epoch']          # the epoch chosen on validation
ck = sorted(glob(f'{OUT}/nllb-lora-final-trainer/checkpoint-*'), key=lambda c: int(c.rsplit('-', 1)[1]))[ep - 1]
CT2 = ct2_export.export(ck, f'{OUT}/nllb-final-ct2-int8')
res = ct2_export.bench(CT2, f'{OUT}/translations', f'{OUT}/results_ct2.json')
print('checkpoint:', ck)
print(res['cpu'], res['cpu_count'], 'cpus, model', res['model_mb'], 'MB')
for r in res['runs']:
    print({k: v for k, v in r.items() if k != 'quality'})
for k, v in res['runs'][-1]['quality'].items():
    print(k, v)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The tokenizer you are loading from '/tmp/nllb-merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


{'threads': 1, 'load_s': 3.9, 'rss_mb_loaded': 2451, 'sentences': 100, 'mean_words': 22.5, 'latency_s_median': 5.1, 'latency_s_p90': 10.04, 'latency_s_mean': 5.53, 'rss_mb_peak': 2627}
{'threads': 2, 'load_s': 3.6, 'rss_mb_loaded': 2452, 'sentences': 100, 'mean_words': 22.5, 'latency_s_median': 2.79, 'latency_s_p90': 5.49, 'latency_s_mean': 3.03, 'rss_mb_peak': 2566}
{'threads': 4, 'load_s': 3.3, 'rss_mb_loaded': 2452, 'sentences': 100, 'mean_words': 22.5, 'latency_s_median': 1.63, 'latency_s_p90': 3.22, 'latency_s_mean': 1.76, 'rss_mb_peak': 2629, 'quality': {'our-test_en2scn': {'fp16 raw': 26.34, 'int8 raw': 26.34, 'fp16 tokenized': 30.13, 'int8 tokenized': 30.09, 'identical': 669, 'lines': 1000, 'batched_s_per_sentence': 0.855}, 'our-test_scn2en': {'fp16 raw': 36.01, 'int8 raw': 35.95, 'fp16 tokenized': 36.58, 'int8 tokenized': 36.52, 'identical': 767, 'lines': 1000, 'batched_s_per_sentence': 0.694}, 'wdowiak-test_en2it': {'fp16 raw': 48.2, 'int8 raw': 47.67, 'fp16 tokenized': 48.77